# **Home Exercise 1 on Text Classification**
Implement a Recurrent Neural Network model (Vanilla RNN, GRU, and LSTM) to predict whether a review is positive or negative.

**Data**: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (the last 10% of rows serve as the test set).
Compare the performance of the three models.


In [1]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import os
import sys
import numpy as np
import pandas as pd

import re
from collections import Counter
import matplotlib.pyplot as plt
from datetime import datetime

print("The last time this project was run is:", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))


if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using device: CUDA - {gpu_name}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    device = torch.device("cpu")
    print("Using device: CPU (CUDA is not available)")


The last time this project was run is: 17:39:11 20/11/2025
Using device: CUDA - Tesla T4
CUDA Capability: (7, 5)


### Download dataset

In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [19]:
!ls /kaggle/input/imdb-dataset-of-50k-movie-reviews

'IMDB Dataset.csv'


In [3]:
# Hyperparameter
MAX_VOCAB_SIZE = 25000
MAX_SEQ_LEN = 200
BATCH_SIZE = 64
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5

## Loading data with DataLoader into DataFrame

In [21]:
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [22]:
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [27]:
from sklearn.model_selection import train_test_split

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f"Tổng số mẫu dữ liệu: {len(df)}")

test_ratio = 0.1
test_size = int(len(df) * test_ratio)

remaining_df = df.iloc[:-test_size]
test_df = df.iloc[-test_size:]


train_df, val_df = train_test_split(
    remaining_df, 
    test_size=1/9,
    random_state=42,
    shuffle=True,
    stratify=remaining_df['sentiment']
)

print("-" * 30)
print(f"Train Set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val Set:   {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test Set:  {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print("-" * 30)

Tổng số mẫu dữ liệu: 50000
------------------------------
Train Set: 40000 samples (80.0%)
Val Set:   5000 samples (10.0%)
Test Set:  5000 samples (10.0%)
------------------------------


### Preprocessing data

In [23]:
import nltk
from tqdm import tqdm
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [15]:
def preprocessing(text):
    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text()
    
    # Remove URL and Email
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S*@\S*\s?', '', text)
    
    # Remove uncommon character
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Lowercase
    text = text.lower()
    
    # Split into tokens
    tokens = text.split()
    
    # Remove stopwords and lemmatization
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    if 'not' in stop_words:
        stop_words.remove('not')
        
    clean_tokens = [
        lemmatizer.lemmatize(token) 
        for token in tokens 
        if token not in stop_words and len(token) > 2 # Bỏ từ quá ngắn
    ]
    
    return clean_tokens

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab=None, max_len=MAX_SEQ_LEN, is_train=True, max_vocab_size=MAX_VOCAB_SIZE):
        """
        Args:
            texts (list/np.array): Danh sách các câu review gốc.
            labels (list/np.array): Danh sách nhãn (0 hoặc 1).
            vocab (dict, optional): Từ điển ánh xạ word->index. Nếu là None (tập train), sẽ tự build.
            max_len (int): Độ dài cố định của chuỗi vector.
            is_train (bool): Cờ đánh dấu là tập train hay test.
        """
        self.labels = torch.tensor(labels, dtype=torch.float)
        self.max_len = max_len
        
        # Step 1: Data preprocessing
        self.processed_texts = [preprocessing(text) for text in tqdm(texts)]
        
        # Step 2: Building vocabulary
        if is_train:
            self.vocab = self.build_vocab(self.processed_texts, max_vocab_size)
        else:
            if vocab is None:
                raise ValueError("Dataset Test/Val can phai duoc cung cap vocab tu tap Train!")
            self.vocab = vocab
            
        # Step 3: Vectorization
        self.sequences = [self.text_to_sequence(tokens) for tokens in self.processed_texts]
            
    def build_vocab(self, processed_texts, max_vocab_size):
        all_tokens = []
        for tokens in processed_texts:
            all_tokens.extend(tokens)
        
        count = Counter(all_tokens)
        sorted_words = count.most_common(max_vocab_size)
        
        # <PAD> = 0, <UNK> = 1
        vocab = {w: i+2 for i, (w, c) in enumerate(sorted_words)}
        vocab['<PAD>'] = 0
        vocab['<UNK>'] = 1
        print(f"Vocabulary size: {len(vocab)}")
        return vocab
    
    def text_to_sequence(self, tokens):
        """
        Chuyển list các tokens thành list các indices
        """
        seq = [self.vocab.get(token, 1) for token in tokens]

        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
            
        return torch.tensor(seq, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]